In [1]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.sql import text
import warnings
import sys
import os # Esto sube un nivel en las carpetas para encontrar la raíz del proyecto
from dotenv import load_dotenv

sys.path.append(os.path.abspath(".."))

import src.db_builder as db

# Verificar dónde está buscando el .env
print("Directorio actual:", os.getcwd())

# Cargar y comprobar variables
load_dotenv("../.env")

Directorio actual: c:\Users\fabih\Desktop\CREATOR\data\data_e-commerce\e-commerce-operations-insights\notebooks


True

In [2]:
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None) # para poder visualizar todas las columnas de los DataFrames

In [3]:
df_data = pd.read_csv("../files/processed/dataco_supply_chain.csv", sep=None, engine="python", encoding="latin-1")

In [4]:
df_data.sample(5)

,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_fname,customer_id,customer_lname,customer_segment,customer_state,customer_street,customer_zipcode,department_id,department_name,latitude,longitude,order_city,order_country,order_customer_id,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_status,product_card_id,product_category_id,product_name,product_price,shipping_mode,delivery_date,order_date,order_time
22034,DEBIT,3,4,52.44,111.58,Advance shipping,0,17,Cleats,Caguas,Puerto Rico,Catherine,5904,Peterson,Consumer,PR,3826 Umber Branch View,725,4,Apparel,18.222580,-66.370506,Quimper,Francia,5904,18777,365,8.4,0.07,46953,59.99,0.47,2,119.98,111.58,52.44,Western Europe,COMPLETE,365,17,Perfect Fitness Perfect Rip Deck,59.99,Standard Class,10-05-2015,10-02-2015,02:04
45110,PAYMENT,2,4,604.80,1260.00,Advance shipping,0,64,Computers,Phoenix,EE. UU.,Leandra,14064,Sullivan,Corporate,AZ,516 Dewy Bluff Swale,85022,10,Technology,33.647820,-112.051880,Schiltigheim,Francia,14064,70511,1351,240.0,0.16,173826,1500.00,0.48,1,1500.00,1260.00,604.80,Western Europe,PENDING_PAYMENT,1351,64,Dell Laptop,1500.00,Standard Class,10-28-2017,10-26-2017,06:47
148575,DEBIT,2,4,30.74,90.95,Advance shipping,0,38,Kids' Golf Clubs,Caguas,Puerto Rico,Mary,1431,Barrera,Home Office,PR,285 Round Bend,725,6,Outdoors,18.255007,-66.370621,BerlÃ­n,Alemania,1431,66081,295,9.0,0.09,165163,99.95,0.34,1,99.95,90.95,30.74,Western Europe,COMPLETE,295,38,Fitbit The One Wireless Activity & Sleep Trac,99.95,Standard Class,08-24-2017,08-22-2017,14:46
74698,DEBIT,2,1,-118.31,173.99,Late delivery,1,48,Water Sports,Caguas,Puerto Rico,Donald,10021,Davila,Consumer,PR,5292 Colonial Grounds,725,7,Fan Shop,18.256407,-66.370506,Cairo,Egipto,10021,42441,1073,26.0,0.13,105961,199.99,-0.68,1,199.99,173.99,-118.31,North Africa,COMPLETE,1073,48,Pelican Sunstream 100 Kayak,199.99,First Class,09-13-2016,09-11-2016,12:37
89653,PAYMENT,2,4,33.06,197.98,Advance shipping,0,9,Cardio Equipment,Virginia Beach,EE. UU.,Harry,4876,Smith,Consumer,VA,5723 Emerald Downs,23456,3,Footwear,36.781036,-76.112663,Buenos Aires,Argentina,4876,2251,191,2.0,0.01,5638,99.99,0.17,2,199.98,197.98,33.06,South America,PAYMENT_REVIEW,191,9,Nike Men's Free 5.0+ Running Shoe,99.99,Standard Class,02-04-2015,02-02-2015,20:16


In [5]:
dim_products = df_data[['product_card_id', 'product_name', 'product_price', 
                        'category_id', 'category_name', 'department_id', 
                        'department_name']].drop_duplicates(subset=['product_card_id']).reset_index(drop=True)

In [6]:
dim_products.tail(5)

,product_card_id,product_name,product_price,category_id,category_name,department_id,department_name
113,646,Merrell Women's Grassbow Sport Hiking Shoe,99.99,30,Men's Golf Clubs,6,Outdoors
114,1361,Toys,11.54,74,Toys,7,Fan Shop
115,1073,Pelican Sunstream 100 Kayak,199.99,48,Water Sports,7,Fan Shop
116,1059,Pelican Maverick 100X Kayak,349.99,48,Water Sports,7,Fan Shop
117,1014,O'Brien Men's Neoprene Life Vest,49.98,46,Indoor/Outdoor Games,7,Fan Shop


In [7]:
dim_customers = df_data[['customer_id', 'customer_fname', 'customer_lname', 
                    'customer_segment', 'customer_street', 
                    'customer_city', 'customer_state', 'customer_zipcode', 'customer_country'
                    ]].drop_duplicates(subset=['customer_id']).reset_index(drop=True)

In [8]:
dim_customers.head(5)

,customer_id,customer_fname,customer_lname,customer_segment,customer_street,customer_city,customer_state,customer_zipcode,customer_country
0,20755,Cally,Holloway,Consumer,5365 Noble Nectar Island,Caguas,PR,725,Puerto Rico
1,19492,Irene,Luna,Consumer,2679 Rustic Loop,Caguas,PR,725,Puerto Rico
2,19491,Gillian,Maldonado,Consumer,8510 Round Bear Gate,San Jose,CA,95125,EE. UU.
3,19490,Tana,Tate,Home Office,3200 Amber Bend,Los Angeles,CA,90027,EE. UU.
4,19489,Orli,Hendricks,Corporate,8671 Iron Anchor Corners,Caguas,PR,725,Puerto Rico


In [9]:
dim_stores = df_data[['latitude', 'longitude', 'shipping_mode']].drop_duplicates().reset_index(drop=True)
dim_stores['store_id'] = dim_stores.index + 1
dim_stores = dim_stores[['store_id', 'latitude', 'longitude', 'shipping_mode']]

In [10]:
dim_stores.sample(5)

,store_id,latitude,longitude,shipping_mode
12911,12912,18.233334,-66.370529,Standard Class
1942,1943,18.237600,-66.370506,Second Class
10557,10558,18.245821,-66.370567,Standard Class
15778,15779,18.245676,-66.370590,Same Day
14769,14770,18.296942,-66.370506,First Class


In [11]:
fact_sales = pd.merge(df_data, dim_stores, on=['latitude', 'longitude', 'shipping_mode'], how='left')

In [12]:
fact_sales.sample(5)

,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_fname,customer_id,customer_lname,customer_segment,customer_state,customer_street,customer_zipcode,department_id,department_name,latitude,longitude,order_city,order_country,order_customer_id,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_status,product_card_id,product_category_id,product_name,product_price,shipping_mode,delivery_date,order_date,order_time,store_id
8426,DEBIT,5,4,-4.67,122.84,Late delivery,1,18,Men's Footwear,Brooklyn,EE. UU.,Margaret,3997,Walker,Corporate,NY,5731 Cinder Horse Court,11218,4,Apparel,40.640594,-73.975189,Chattanooga,Estados Unidos,3997,40034,403,7.15,0.06,99891,129.99,-0.04,1,129.99,122.84,-4.67,South of USA,COMPLETE,403,18,Nike Men's CJ Elite 2 TD Football Cleat,129.99,Standard Class,08-12-2016,08-07-2016,09:20,300
47371,PAYMENT,4,4,64.02,176.37,Shipping on time,0,17,Cleats,Far Rockaway,EE. UU.,Mary,10164,Parsons,Corporate,NY,2 Foggy Corners,11691,4,Apparel,42.088577,-76.888535,Ulan Bator,Mongolia,10164,45513,365,3.60,0.02,113736,59.99,0.36,3,179.97,176.37,64.02,Eastern Asia,PENDING_PAYMENT,365,17,Perfect Fitness Perfect Rip Deck,59.99,Standard Class,10-30-2016,10-26-2016,08:52,3187
40263,DEBIT,6,4,10.40,129.99,Late delivery,1,18,Men's Footwear,Caguas,Puerto Rico,Eric,2833,Vega,Corporate,PR,4377 Heather Canyon,725,4,Apparel,18.203897,-66.370544,Cairo,Egipto,2833,51070,403,0.00,0.00,127645,129.99,0.08,1,129.99,129.99,10.40,North Africa,COMPLETE,403,18,Nike Men's CJ Elite 2 TD Football Cleat,129.99,Standard Class,01-21-2017,01-15-2017,11:44,12322
56918,DEBIT,2,4,-2.29,176.37,Advance shipping,0,17,Cleats,Caguas,Puerto Rico,Mary,11305,Hartman,Consumer,PR,5193 Foggy Apple Jetty,725,4,Apparel,18.232094,-66.370590,Los Angeles,Estados Unidos,11305,37071,365,3.60,0.02,92496,59.99,-0.01,3,179.97,176.37,-2.29,West of USA,COMPLETE,365,17,Perfect Fitness Perfect Rip Deck,59.99,Standard Class,06-27-2016,06-25-2016,03:16,8094
63320,CASH,5,4,-16.63,47.50,Late delivery,1,24,Women's Apparel,Harvey,EE. UU.,David,2113,Smith,Home Office,IL,5902 Silver Embers Diversion,60426,5,Golf,41.786617,-90.132309,BerlÃ­n,Alemania,2113,13824,502,2.50,0.05,34632,50.00,-0.35,1,50.00,47.50,-16.63,Western Europe,CLOSED,502,24,Nike Men's Dri-FIT Victory Golf Polo,50.00,Standard Class,07-26-2015,07-21-2015,18:48,10715


In [13]:
fact_sales = fact_sales[[
    'order_item_id', 'order_id', 'customer_id', 'product_card_id', 'store_id',
    'order_date_dateorders',
    'type', 'order_status', 'days_for_shipping_real', 
    'days_for_shipment_scheduled', 'delivery_status', 'late_delivery_risk',
    'order_city', 'order_country', 'order_region',
    'sales', 'order_item_quantity', 'order_item_product_price', 
    'order_item_discount', 'order_item_discount_rate', 'order_item_total', 
    'order_item_profit_ratio', 'benefit_per_order', 'order_profit_per_order', 'sales_per_customer'
]]
 
print("¡DataFrames del Modelo en Estrella listos en memoria!")

KeyError: "['order_date_dateorders'] not in index"

In [ ]:
USER  = os.getenv("DB_USER")
PASS  = os.getenv("DB_PASSWORD")
HOST  = os.getenv("DB_HOST")
PORT  = os.getenv("DB_PORT", "3306")   # 3306 como valor por defecto
DB_NAME = os.getenv("DB_NAME")
 
# 1️⃣ Conectar al servidor SIN base de datos

engine_server = create_engine(f"mysql+pymysql://{USER}:{PASS}@{HOST}:{PORT}") 
# 2️⃣ Crear la base de datos si no existe

with engine_server.begin() as con:
    con.execute(text(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}"))
    print(f"✅ Base de datos '{DB_NAME}' lista")
 
engine_server.dispose()
 
# 3️⃣ Conectar ahora CON la base de datos

engine = create_engine(f"mysql+pymysql://{USER}:{PASS}@{HOST}:{PORT}/{DB_NAME}")
print(f"✅ Conectado a '{DB_NAME}'")
 

✅ Base de datos 'dataco' lista
✅ Conectado a 'dataco'


In [ ]:
# Carga del DataFrame en MySQL
# Esto crea automáticamente la tabla products en la base de datos
 
dim_products.to_sql("products", con=engine, if_exists="replace", index=False)
 
'''# Comprobación previa para evitar errores al asignar la clave primaria
dim_products = dim_products.dropna(subset=["Employee_Number"])
dim_products = dim_products.drop_duplicates(subset=["Employee_Number"])'''
 
dim_products.to_sql("products", con=engine, if_exists="replace", index=False)
 
# Asignación de la clave primaria una vez creada la tabla
with engine.begin() as con:
    con.execute(text("""
        ALTER TABLE products
        MODIFY product_card_id INT NOT NULL"""))
    con.execute(text("""
        ALTER TABLE products
        ADD PRIMARY KEY (product_card_id)"""))
 
print("¡Listo! Base de datos, tabla y clave primaria creadas correctamente.")

¡Listo! Base de datos, tabla y clave primaria creadas correctamente.
